In [1]:
import pathlib
from datetime import datetime
import torch

from artist.data_parser import paint_scenario_parser
from artist.scenario.configuration_classes import (
    LightSourceConfig,
    LightSourceListConfig,
)
from artist.scenario.h5_scenario_generator import H5ScenarioGenerator
from artist.util import config_dictionary, set_logger_config
from artist.util.environment_setup import get_device

# Set up logger
set_logger_config()

torch.manual_seed(7)
torch.cuda.manual_seed(7)

# Set the device.
device = get_device()

# Specify the path to your paint dataset folder
paint_dataset_path = pathlib.Path("../paint_dataset")

# Specify the path where you want to save the scenario file
scenario_path = pathlib.Path("../scenarios/all_heliostats_scenario/all_heliostats_scenario.h5")

# Specify the path to your tower-measurements.json file
tower_file = pathlib.Path("../paint_dataset/WRI1030197-tower-measurements.json")


[2026-02-03 23:31:28,138][artist.util.environment_setup][INFO] - No device type provided. The device will default to GPU based on availability and OS, otherwise to CPU.
[2026-02-03 23:31:28,139][artist.util.environment_setup][WARNING] - Setting device to CPU. ARTIST only supports CPU for MacOS.


In [2]:
def find_latest_deflectometry_file(heliostat_name: str, deflectometry_folder: pathlib.Path) -> pathlib.Path | None:
    """
    Find the latest deflectometry file for a given heliostat.
    
    Parameters
    ----------
    heliostat_name : str
        Name of the heliostat (e.g., "AA31")
    deflectometry_folder : pathlib.Path
        Path to the Deflectometry folder
    
    Returns
    -------
    pathlib.Path | None
        Path to the latest deflectometry file, or None if not found
    """
    # Pattern: {heliostat_name}-filled-*-deflectometry.h5
    pattern = f"{heliostat_name}-filled-*-deflectometry.h5"
    deflectometry_files = list(deflectometry_folder.glob(pattern))
    
    if not deflectometry_files:
        return None
    
    # Extract datetime from filename and find the latest
    # Format: AA31-filled-2021-10-13Z09-29-29Z-deflectometry.h5
    def extract_datetime(filepath: pathlib.Path) -> datetime:
        parts = filepath.stem.split('-')
        # Find the date part (YYYY-MM-DD)
        for i, part in enumerate(parts):
            if len(part) == 4 and part.isdigit():  # Year
                date_str = f"{parts[i]}-{parts[i+1]}-{parts[i+2].split('Z')[0]}"
                time_str = f"{parts[i+2].split('Z')[1]}-{parts[i+3]}-{parts[i+4].split('Z')[0]}"
                datetime_str = f"{date_str} {time_str.replace('-', ':')}"
                return datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S")
        return datetime.min
    
    latest_file = max(deflectometry_files, key=extract_datetime)
    return latest_file

def build_heliostat_files_list(paint_dataset_path: pathlib.Path) -> list[tuple[str, pathlib.Path, pathlib.Path]]:
    """
    Build the heliostat files list by scanning the paint_dataset folder.
    
    Parameters
    ----------
    paint_dataset_path : pathlib.Path
        Path to the paint_dataset folder
    
    Returns
    -------
    list[tuple[str, pathlib.Path, pathlib.Path]]
        List of tuples: (heliostat_name, properties_path, deflectometry_path)
    """
    heliostat_files_list = []
    
    # Iterate through all subdirectories in paint_dataset
    for heliostat_folder in sorted(paint_dataset_path.iterdir()):
        if not heliostat_folder.is_dir():
            continue
        
        heliostat_name = heliostat_folder.name
        
        # Skip non-heliostat folders (like the tower measurements file)
        if not heliostat_name.startswith(('AA', 'AB', 'AC', 'AD', 'BA', 'BB', 'BC', 'BD')):  # Adjust prefixes as needed
            continue
        
        properties_folder = heliostat_folder / "Properties"
        deflectometry_folder = heliostat_folder / "Deflectometry"
        
        # Check if required folders exist
        if not properties_folder.exists() or not deflectometry_folder.exists():
            print(f"Warning: Skipping {heliostat_name} - missing Properties or Deflectometry folder")
            continue
        
        # Find properties file
        properties_pattern = f"{heliostat_name}-heliostat-properties.json"
        properties_files = list(properties_folder.glob(properties_pattern))
        
        if not properties_files:
            print(f"Warning: Skipping {heliostat_name} - properties file not found")
            continue
        
        properties_file = properties_files[0]
        
        # Find latest deflectometry file
        deflectometry_file = find_latest_deflectometry_file(heliostat_name, deflectometry_folder)
        
        if deflectometry_file is None:
            print(f"Warning: Skipping {heliostat_name} - no deflectometry file found")
            continue
        
        heliostat_files_list.append((heliostat_name, properties_file, deflectometry_file))
        print(f"Added {heliostat_name}: {deflectometry_file.name}")
    
    return heliostat_files_list


In [3]:
# Build the heliostat files list automatically
heliostat_files_list = build_heliostat_files_list(paint_dataset_path)

print(f"\nFound {len(heliostat_files_list)} heliostats:")
for name, _, deflect in heliostat_files_list:
    print(f"  - {name}: {deflect.name}")

# Check if we found any heliostats
if not heliostat_files_list:
    raise ValueError("No heliostats found in the paint_dataset folder!")

# Check if the scenario save path is valid
if not scenario_path.parent.exists():
    print(f"Creating directory: {scenario_path.parent}")
    scenario_path.parent.mkdir(parents=True, exist_ok=True)

# Include the power plant configuration
power_plant_config, target_area_list_config = (
    paint_scenario_parser.extract_paint_tower_measurements(
        tower_measurements_path=tower_file, device=device
    )
)

# Include the light source configuration
light_source1_config = LightSourceConfig(
    light_source_key="sun_1",
    light_source_type=config_dictionary.sun_key,
    number_of_rays=10,
    distribution_type=config_dictionary.light_source_distribution_is_normal,
    mean=0.0,
    covariance=4.3681e-06,
)

# Create a list of light source configs - in this case only one
light_source_list = [light_source1_config]

# Include the configuration for the list of light sources
light_source_list_config = LightSourceListConfig(light_source_list=light_source_list)

target_area = [
    target_area
    for target_area in target_area_list_config.target_area_list
    if target_area.target_area_key == config_dictionary.target_area_receiver
]

# NURBS fitting parameters
number_of_nurbs_control_points = torch.tensor([20, 20], device=device)
nurbs_fit_method = config_dictionary.fit_nurbs_from_normals
nurbs_deflectometry_step_size = 100
nurbs_fit_tolerance = 1e-10
nurbs_fit_max_epoch = 400

# Please leave the optimizable parameters empty, they will automatically be added for the surface fit
nurbs_fit_optimizer = torch.optim.Adam([torch.empty(1, requires_grad=True)], lr=1e-3)
nurbs_fit_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    nurbs_fit_optimizer,
    mode="min",
    factor=0.2,
    patience=50,
    threshold=1e-7,
    threshold_mode="abs",
)

# Use this configuration for deflectometry surfaces
print("\nFitting NURBS surfaces from deflectometry data...")
heliostat_list_config, prototype_config = (
    paint_scenario_parser.extract_paint_heliostats_fitted_surface(
        paths=heliostat_files_list,
        power_plant_position=power_plant_config.power_plant_position,
        number_of_nurbs_control_points=number_of_nurbs_control_points,
        deflectometry_step_size=nurbs_deflectometry_step_size,
        nurbs_fit_method=nurbs_fit_method,
        nurbs_fit_tolerance=nurbs_fit_tolerance,
        nurbs_fit_max_epoch=nurbs_fit_max_epoch,
        nurbs_fit_optimizer=nurbs_fit_optimizer,
        nurbs_fit_scheduler=nurbs_fit_scheduler,
        device=device,
    )
)

Added AA23: AA23-filled-2021-10-13Z09-27-07Z-deflectometry.h5
Added AA24: AA24-filled-2021-10-13Z09-29-29Z-deflectometry.h5
Added AA25: AA25-filled-2021-10-13Z09-32-36Z-deflectometry.h5
Added AA26: AA26-filled-2021-10-13Z09-34-21Z-deflectometry.h5
Added AA27: AA27-filled-2021-10-13Z09-36-03Z-deflectometry.h5
Added AA28: AA28-filled-2023-09-18Z11-43-43Z-deflectometry.h5
Added AA29: AA29-filled-2021-10-12Z13-30-29Z-deflectometry.h5
Added AA30: AA30-filled-2023-09-18Z11-48-45Z-deflectometry.h5
Added AA31: AA31-filled-2023-09-18Z11-50-48Z-deflectometry.h5
Added AA32: AA32-filled-2021-10-12Z13-31-49Z-deflectometry.h5
Added AA33: AA33-filled-2021-10-12Z13-38-32Z-deflectometry.h5
Added AA34: AA34-filled-2021-10-12Z13-38-59Z-deflectometry.h5
Added AA35: AA35-filled-2023-09-18Z11-51-43Z-deflectometry.h5
Added AA36: AA36-filled-2021-10-12Z13-42-51Z-deflectometry.h5
Added AA38: AA38-filled-2016-06-30Z13-36-50Z-deflectometry.h5
Added AA39: AA39-filled-2023-09-18Z08-49-09Z-deflectometry.h5
Added AA

In [4]:

"""Generate the scenario given the defined parameters."""
print(f"\nGenerating scenario at: {scenario_path}")
scenario_generator = H5ScenarioGenerator(
    file_path=scenario_path,
    power_plant_config=power_plant_config,
    target_area_list_config=target_area_list_config,
    light_source_list_config=light_source_list_config,
    prototype_config=prototype_config,
    heliostat_list_config=heliostat_list_config,
)
scenario_generator.generate_scenario()
print(f"Scenario successfully generated with {len(heliostat_files_list)} heliostats!")


Generating scenario at: ../scenarios/all_heliostats_scenario/all_heliostats_scenario.h5
[2026-01-22 11:01:12,595][artist.scenario.h5_scenario_generator][INFO] - Generating a scenario saved to: ../scenarios/all_heliostats_scenario/all_heliostats_scenario.h5.
[2026-01-22 11:01:12,600][artist.scenario.h5_scenario_generator][INFO] - Using scenario generator version 1.0.
[2026-01-22 11:01:12,603][artist.scenario.h5_scenario_generator][INFO] - Including parameters for the power plant.
[2026-01-22 11:01:12,605][artist.scenario.h5_scenario_generator][INFO] - Including parameters for the target areas.
[2026-01-22 11:01:12,608][artist.scenario.h5_scenario_generator][INFO] - Including parameters for the light sources.
[2026-01-22 11:01:12,609][artist.scenario.h5_scenario_generator][INFO] - Including parameters for the prototype.
[2026-01-22 11:01:12,614][artist.scenario.h5_scenario_generator][INFO] - Including parameters for the heliostats.
Scenario successfully generated with 137 heliostats!


In [4]:
print([name for name, _, _ in heliostat_files_list])
heliostat_ids = [name for name, _, _ in heliostat_files_list]

['AA23', 'AA24', 'AA25', 'AA26', 'AA27', 'AA28', 'AA29', 'AA30', 'AA31', 'AA32', 'AA33', 'AA34', 'AA35', 'AA36', 'AA38', 'AA39', 'AA44', 'AA49', 'AA50', 'AB26', 'AB27', 'AB29', 'AB30', 'AB32', 'AB33', 'AB34', 'AB35', 'AB36', 'AB37', 'AB38', 'AB39', 'AB40', 'AB41', 'AB42', 'AB43', 'AB44', 'AB45', 'AB46', 'AB47', 'AB50', 'AC24', 'AC25', 'AC26', 'AC27', 'AC28', 'AC29', 'AC30', 'AC31', 'AC32', 'AC33', 'AC34', 'AC35', 'AC36', 'AC37', 'AC38', 'AC39', 'AC40', 'AC41', 'AC42', 'AC43', 'AC45', 'AC47', 'AC48', 'AD29', 'AD30', 'AD31', 'AD33', 'AD34', 'AD35', 'AD36', 'AD37', 'AD39', 'AD40', 'AD42', 'AD43', 'AD44', 'AD45', 'AD46', 'BA27', 'BA28', 'BA29', 'BA35', 'BA36', 'BA37', 'BA38', 'BA39', 'BA40', 'BA41', 'BA42', 'BA43', 'BA44', 'BA61', 'BA62', 'BA63', 'BA65', 'BA66', 'BA69', 'BA70', 'BA71', 'BA72', 'BA73', 'BB21', 'BB28', 'BB29', 'BB30', 'BB37', 'BB39', 'BB40', 'BB71', 'BB72', 'BC17', 'BC19', 'BC28', 'BC32', 'BC33', 'BC34', 'BC35', 'BC60', 'BC61', 'BC62', 'BC63', 'BC64', 'BC66', 'BC71', 'BD28',

In [5]:
import shutil

# Create the paint_files directory
paint_files_dir = pathlib.Path("../paint_files")
paint_files_dir.mkdir(exist_ok=True)

print(f"Creating paint_files directory at: {paint_files_dir.absolute()}\n")

# Copy files for each heliostat
for heliostat_name, properties_file, deflectometry_file in heliostat_files_list:
    # Create heliostat subfolder
    heliostat_dir = paint_files_dir / heliostat_name
    heliostat_dir.mkdir(exist_ok=True)
    
    # Copy properties file
    properties_dest = heliostat_dir / properties_file.name
    shutil.copy2(properties_file, properties_dest)
    
    # Copy deflectometry file
    deflectometry_dest = heliostat_dir / deflectometry_file.name
    shutil.copy2(deflectometry_file, deflectometry_dest)
    
    print(f"✓ {heliostat_name}: Copied properties and deflectometry files")

print(f"\nSuccessfully created {len(heliostat_files_list)} heliostat folders in {paint_files_dir.absolute()}")

Creating paint_files directory at: /Users/alexandru/Master Thesis/master-thesis/src/../paint_files

✓ AA23: Copied properties and deflectometry files
✓ AA24: Copied properties and deflectometry files
✓ AA25: Copied properties and deflectometry files
✓ AA26: Copied properties and deflectometry files
✓ AA27: Copied properties and deflectometry files
✓ AA28: Copied properties and deflectometry files
✓ AA29: Copied properties and deflectometry files
✓ AA30: Copied properties and deflectometry files
✓ AA31: Copied properties and deflectometry files
✓ AA32: Copied properties and deflectometry files
✓ AA33: Copied properties and deflectometry files
✓ AA34: Copied properties and deflectometry files
✓ AA35: Copied properties and deflectometry files
✓ AA36: Copied properties and deflectometry files
✓ AA38: Copied properties and deflectometry files
✓ AA39: Copied properties and deflectometry files
✓ AA44: Copied properties and deflectometry files
✓ AA49: Copied properties and deflectometry files
